# Day 1 — Solution: OLS from Scratch

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices, synthetic_returns

## E1 — `ols()` against the planted truth

In [ ]:
def ols(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    X = np.column_stack([np.ones(len(x)), x])
    return np.linalg.solve(X.T @ X, X.T @ y)

rng = np.random.default_rng(7)
m = rng.normal(0.0005, 0.010, 1500)
e = rng.normal(0, 0.008, 1500)
y = 0.0002 + 1.20 * m + e

a_hat, b_hat = ols(m, y)
# quick SE on beta, closed form (day 3 formalizes): sigma_e / (sigma_m * sqrt(n))
resid = y - a_hat - b_hat * m
se_b = resid.std(ddof=2) / (m.std(ddof=1) * np.sqrt(len(m)))
print(f"alpha_hat {a_hat:.5f} (truth 0.00020) | beta_hat {b_hat:.4f} (truth 1.20)")
print(f"beta error {b_hat - 1.20:+.4f} vs ~SE {se_b:.4f} -> z {(b_hat-1.20)/se_b:+.2f}")

**Expected reasoning.** β̂ lands within ~1–2 SE of 1.20 (z well under 2 in
magnitude) and α̂ within noise of 0.0002 — **the estimator recovers the
planted world, which is the entire reason we planted one.** Notice what the
exercise quietly taught: *unbiased* does not mean *exact* — 1,500 days and
β̂ still wobbles by ±0.02. That wobble is day 3's subject, and it is why
every beta you ever quote needs its SE attached.

## E2 — against statsmodels, to machine precision

In [ ]:
import statsmodels.api as sm
fit = sm.OLS(y, sm.add_constant(m)).fit()
params = np.asarray(fit.params)
diff = np.abs(params - np.array([a_hat, b_hat])).max()
print(f"statsmodels: alpha {params[0]:.5f} beta {params[1]:.4f}")
print(f"max |hand - statsmodels| = {diff:.2e}")

**Expected pattern.** `diff` ≈ 1e-15 to 1e-17 — float noise. **The library
was never doing anything you can't derive; it just does it at scale, with
correct bookkeeping.** From here on you are allowed statsmodels — you have
earned it — but the SEs it prints remain your responsibility (days 4–5, 8).

## E3 — QQQ on the market

In [ ]:
if DATA_SOURCE == "real":
    px = get_prices(["QQQ", "SPY"], start="2010-01-01")
else:
    px = synthetic_prices(n_days=3800, n_assets=2, seed=11, corr=0.64,
                          drift_spread=0.0001)
    px.columns = ["QQQ", "SPY"]
r = np.log(px).diff().dropna()
a, b = ols(r["SPY"], r["QQQ"])
corr = r["QQQ"].corr(r["SPY"])
print(f"QQQ ~ SPY:  alpha {a*252*100:+.2f}%/yr   beta {b:.3f}   corr {corr:+.2f}")
print(f"closed form check: corr*sd(y)/sd(x) = {corr*r['QQQ'].std()/r['SPY'].std():.3f}")

**Expected pattern.** Real world: β ≈ 1.1–1.3 — QQQ is a levered position
in the market factor, i.e. holding QQQ while believing yourself diversified
is holding ~1.2 units of market risk. The closed-form check matches to
displayed precision: **β is correlation wearing a volatility ratio.** α
annualized is small and (module 04 discipline) carries an error bar several
times its own size — the correct reading is "consistent with zero", not a
number to trade. Synthetic mode: recovers corr ≈ 0.64 with per-asset vol
spread, so β ≈ 0.64·σ_QQQ/σ_SPY — your call which two assets the synthetic
world made.

## E4 — broken, informatively

In [ ]:
# (a) no intercept
X1 = m.reshape(-1, 1)
b_noint = np.linalg.solve(X1.T @ X1, X1.T @ y)[0]
print(f"(a) beta without intercept: {b_noint:.4f} vs with intercept {b_hat:.4f}")

# (b) levels regression on random-walk prices
p_m = 100 * np.exp(np.cumsum(m))
p_y = 100 * np.exp(np.cumsum(y))
a_lvl, b_lvl = ols(p_m, p_y)
r2_lvl = np.corrcoef(p_m, p_y)[0, 1] ** 2
e_lvl = p_y - a_lvl - b_lvl * p_m
print(f"(b) levels: beta {b_lvl:.3f}, R2 {r2_lvl:.3f}, resid lag-1 autocorr {e_lvl[1:] @ e_lvl[:-1] / (e_lvl @ e_lvl):+.3f}")

**Expected reasoning.** (a) The no-intercept slope moves *toward
E[xy]/E[x²]*; with a nonzero mean in y, part of the level of y is loaded
into β — the bias appears *in the slope*, not the (deleted) intercept.
Asserting α = 0 is a modeling claim with a price. (b) Levels of two
integrated series give R² ≈ 0.9+ and β̂ that reflects share-of-trend, not
co-movement; the residual autocorrelation (~0.95+) is the tell — **a great
fit whose errors are a smoother random walk than the data.** Rule: high R²
on financial *levels* is a warning light, not a result.

## E5 — the PM's three failure channels (exemplar)

1. **Sampling error** — β̂ is a draw; at 1,500 days its SD is ~0.01–0.02,
   at 60 days ~0.08. For a hedge-ratio decision, window length IS the risk
   parameter.
2. **Regime/window dependence** — the relationship β̂ measures is not a
   constant of nature; estimate it in 2019 and apply it in March 2020 and
   you own the difference (day 7 measures this; day 17 makes it a research
   question).
3. **Omitted structure** — β̂ conditioned only on the market ignores what
   the market itself correlates with (rate shocks, sector beta); if the
   omitted driver moves, your hedge ratio inherits the omission. Days 4–5
   turn this from intuition into a checklist.

**Common mistakes.** Quoting β to two decimals with no window attached;
treating α̂ > 0 on one regression as skill (it is a point estimate with a
wide CI); believing machine-precision agreement with statsmodels (E2) means
the *number* is right — it means the *arithmetic* is right, and arithmetic
was never the hard part.